<a href="https://colab.research.google.com/github/lxndrbnsv/cir-notebooks/blob/main/sort_distance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Устанавливаем зависимости*

In [6]:
%%capture
!pip install openpyxl
!pip install requests

# Константы

*Загружаем исходный файл и прописываем его имя в константу SOURCE_FILE*

In [8]:
SOURCE_FILE = "coords.txt"

*Прописываем имя для файла с результатами*

In [113]:
OUTPUT_FILE = "results.xlsx"

*Добавляем ключ Яндекс ГЕО API*

In [14]:
YANDEX_API_KEY = ""

# Логика

*Импортируем библиотеки*

In [59]:
import math

import requests
import openpyxl

## Сбор координат

*Получение координат с API Яндекса*

In [29]:
def get_coordinates(addr_str: str) -> tuple|None:
  """Отправляем запрос на API Яндекса, чтобы получить координаты."""
  url = "https://geocode-maps.yandex.ru/1.x/"
  params = {"apikey": YANDEX_API_KEY, "format": "json", "geocode": address}
  r = requests.get(url, params=params).json()
  try:
      pos = r["response"]["GeoObjectCollection"]["featureMember"][0][
          "GeoObject"
      ]["Point"]["pos"]
      lat, lon = pos.split()
      return float(lat), float(lon)
  except:
      return None

***Добавляем координаты каждого адреса***

*Получаем адреса из файлов и удаляем возможные дубли*

In [46]:
with open(SOURCE_FILE, "r") as text_file:
  text_data = text_file.readlines()

addresses = list({t.replace("\n", "") for t in text_data})

*Добавляем координаты*

In [102]:
rows = []
for num, address in enumerate(addresses):
  print(
    f"\rПолучение координат адреса {num + 1} из {len(addresses)}",
    end="",
    flush=True
  )
  lat, lon = get_coordinates(address)
  rows.append((address, f"{lat}, {lon}"))

print("\nГотово!")

Получение координат адреса 424 из 424
Готово!


## Сортировка

***Выполняем сортировку***


In [89]:
def calculate_distance(lat1, lon1, lat2, lon2):
    """
    Вычисляет расстояние в километрах между двумя точками на Земле
    по формуле гаверсинусов.
    """
    R = 6371  # радиус Земли в км
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)

    a = math.sin(delta_phi / 2) ** 2 + \
        math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c

*Выстраиваем все адреса в цепочку по расстоянию.*


*Начинаем с первого адреса, каждый следующий - ближайший к предыдущему из оставшихся*

In [122]:

# Создаём словарь координат для быстрого доступа
coords_dict = {}
for addr, coord_str in rows:
    coords_dict[addr] = coord_str

result_chain = []  # список кортежей (адрес, расстояние от предыдущего)
used_addresses = set()  # использованные адреса

# Начинаем с первого адреса из rows
current_address = rows[0][0]
result_chain.append((current_address, 0))
used_addresses.add(current_address)

# Пока не использовали все адреса
while len(used_addresses) < len(rows):
    current_lat, current_lon = map(float, coords_dict[current_address].split(', '))

    # Ищем ближайший неиспользованный адрес
    min_dist = float('inf')
    next_address = None

    for addr, coord_str in rows:
        if addr in used_addresses:
            continue

        # Координаты кандидата
        lat, lon = map(float, coord_str.split(', '))
        dist = calculate_distance(current_lat, current_lon, lat, lon)

        if dist < min_dist:
            min_dist = dist
            next_address = addr

    # Добавляем ближайший найденный адрес
    if next_address:
        result_chain.append((next_address, min_dist))
        used_addresses.add(next_address)
        current_address = next_address

print(f"Построена цепочка из {len(result_chain)} адресов")

Построена цепочка из 424 адресов


# Запись результатов

In [120]:
def write_xlsx(result_chain, filename):
    """
    Записывает результат в Excel файл.
    result_chain: список кортежей (адрес, расстояние)
    filename: имя файла для сохранения
    """
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Результат"

    ws['A1'] = "Адрес"
    ws['B1'] = "Расстояние (км)"

    for row_num, (address, distance) in enumerate(result_chain, start=2):
        ws[f'A{row_num}'] = address
        ws[f'B{row_num}'] = round(distance, 3) if distance != 0 else 0

    wb.save(filename)
    print(f"Файл {filename} успешно сохранён")

In [121]:
write_xlsx(result_chain, OUTPUT_FILE)

Файл results.xlsx успешно сохранён
